# Baseline evaluation on GPU (Colab)
Measures the frozen ArcFace R50 on halfFace = the **before** numbers.

**Before running:** Runtime > Change runtime type > **T4 GPU**. No Google Drive needed: you pick `techfest_code.zip` in cell 4.

In [ ]:
!nvidia-smi -L

In [ ]:
# plain `onnxruntime` (pulled in by insightface) shadows the GPU build, so remove it first
!pip -q uninstall -y onnxruntime onnxruntime-gpu
!pip -q install insightface onnx opencv-python-headless scikit-learn pandas matplotlib tqdm
!pip -q uninstall -y onnxruntime
!pip -q install "onnxruntime-gpu[cuda,cudnn]" || pip -q install onnxruntime-gpu
!pip list 2>/dev/null | grep -i onnxruntime

In [ ]:
# A file picker appears: choose techfest_code.zip from your home folder (27 KB)
from google.colab import files
up = files.upload()
!rm -rf /content/tf && mkdir -p /content/tf && cd /content/tf && unzip -q /content/techfest_code.zip && ls

In [ ]:
# halfFace (CC BY 4.0, Mendeley pk44mkx9vm), ~690 MB
%cd /content/tf
!mkdir -p data/halfface/extracted
!wget -q -O hf.zip "https://data.mendeley.com/public-files/datasets/pk44mkx9vm/files/8d2795bf-aa83-4d58-8037-a5e6994b4aa5/file_downloaded"
!unzip -q hf.zip -d data/halfface/extracted && rm hf.zip
!ls data/halfface/extracted

In [ ]:
import os, pandas as pd
m = pd.read_csv('data/halfface/manifest.csv')
missing = [p for p in m.path if not os.path.exists('data/halfface/'+p)]
print(len(m), 'images in manifest; missing:', len(missing)); print(missing[:3])

In [ ]:
# RUN THIS BEFORE the evaluation cell: is the GPU really used by our models?
import os; os.environ['FACE_PROVIDER']='cuda'
%cd /content/tf
import sys; sys.path.insert(0,'.')
import onnxruntime as ort
print('ORT', ort.__version__, 'available:', ort.get_available_providers())
from pipeline import FacePipeline
p = FacePipeline()
print('detector providers  :', p.app.det_model.session.get_providers())
print('recognizer providers:', p.rec.session.get_providers())
# first entry must be CUDAExecutionProvider. If it says CPUExecutionProvider, ORT silently fell back to CPU.
!nvidia-smi --query-gpu=name,memory.used,utilization.gpu --format=csv

In [ ]:
%cd /content/tf
!FACE_PROVIDER=cuda FACE_WORKERS=1 python evaluate.py --manifest data/halfface/manifest.csv --out results/halfface --ablation 2>&1 | grep -v -i -E "warn|tform" | tail -40

In [ ]:
import onnxruntime as ort; print('providers available:', ort.get_available_providers())  # CUDAExecutionProvider must be listed or the run above used CPU

In [ ]:
!cd /content/tf && zip -qr /content/techfest_baseline_results.zip results
from google.colab import files
files.download('/content/techfest_baseline_results.zip')  # saves to your Downloads folder